# SimLOB Baseline Notebook
This notebook implements and trains the baseline encoder model under our unified, leakage-safe LOBench replication pipeline.

In [1]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

Mounted at /content/drive
Mounted Google Drive and changed directory to baselines.


In [2]:
# Install PyTorch Lightning if it is not present in the environment
!pip install -q lightning pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 61.9 MB/s eta 0:00:00


In [3]:
from common import *
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Libraries and common module imported successfully.')

Libraries and common module imported successfully.


In [4]:
class SimLOBEncoder(nn.Module):
    def __init__(self, n_features=40, d_model=256, nhead=8, num_layers=2,
                 dim_feedforward=512, latent_dim=256, seq_len=100):
        super().__init__()
        # FCN1: per-timestep feature extraction, 40 -> 256
        self.fcn1 = nn.Linear(n_features, d_model)
        # Transformer stack, L=2 per the paper's chosen default
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # FCN2: dimension reduction block -- project back to n_features per timestep,
        # flatten, then reduce to the latent vector
        self.reduce_proj = nn.Linear(d_model, n_features)
        self.fcn2 = nn.Sequential(
            nn.Linear(seq_len * n_features, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim),
        )

    def forward(self, x):  # x: [B, 100, 40]
        h = self.fcn1(x)                        # [B, 100, 256]
        h = self.transformer(h)                  # [B, 100, 256]
        h = self.reduce_proj(h)                  # [B, 100, 40]
        h = h.reshape(h.shape[0], -1)             # [B, 4000]
        return self.fcn2(h)                       # [B, latent_dim]

In [5]:
model_name = 'SimLOB'
stocks = ['sz000001', 'sz000002', 'sz000858', 'sz300147', 'sz002415']
for stock in stocks:
    print(f'\n========================================')
    print(f'Starting experiment for Model: {model_name} | Stock: {stock}')
    print(f'========================================')
    run_experiment(
        encoder_class=SimLOBEncoder,
        model_name=model_name,
        stock=stock,
        latent_dim=256,
        max_epochs=100
    )


Starting experiment for Model: SimLOB | Stock: sz000001
Loading data from data/sz000001-level10_processed.csv...
Loaded shape: (1171534, 41)
Columns and Level 10 ordering verified successfully.
Detecting sessions (3-second continuous chunks)...
Detected 488 sessions.
Splitting train/val/test splits (80/10/10 by calendar dates)...
Constructing sequence window indices (seq_len=100, stride=1)...

--- Train/Val/Test Splits & Windows ---
Split      | Rows       | Window Count
Train      | 936254     | 897644
Validation | 115231     | 110479
Test       | 120049     | 115099
---------------------------------------

Encoder parameters: 5,828,136
Shared Decoder parameters: 4,756,896
Total model parameters: 10,585,032
Found existing checkpoint at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/SimLOB/sz000001/last.ckpt. Resuming training...


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory training_logs/SimLOB/sz000001 exists and is not empty. Previous log files in this directory will be deleted when the new ones

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ SimLOBEncoder │  5.8 M │ train │     0 │
│ 1 │ decoder │ SharedDecoder │  4.8 M │ train │     0 │
│ 2 │ loss_fn │ MSELoss       │      0 │ train │     0 │
│ 3 │ mae_fn  │ L1Loss        │      0 │ train │     0 │
└───┴─────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 10.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.6 M                                                                                               
Total estimated model params size (MB): 42.340                                                                     
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO: Restored all states from the checkpoint at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/SimLOB/sz000001/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restored all states from the checkpoint at /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/SimLOB/sz000001/last.ckpt


...

Training loop finished in 14287 seconds.
Loading best checkpoint for evaluation: /content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/SimLOB/sz002415/best-v1.ckpt


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │    0.09370214492082596    │
│         test_mse          │    0.05030078813433647    │
└───────────────────────────┴───────────────────────────┘


=== FINAL REPORT ===
Model: SimLOB | Stock: sz002415
Encoder params: 5,828,136
Total params (encoder + shared decoder): 10,585,032
Training time: 14287s
Test MSE: 0.0503
Test MAE: 0.0937

